## 01 — Live Layer Switching

Everything comes together here.

We wire three things into one map:
1. **LOD selection** — `get_lod(zoom)` picks which dataset to use
2. **Grid index culling** — the correct grid index is queried for the current viewport
3. **Event handling** — zoom and pan events trigger the right updates

After this notebook, we have a working map that automatically serves the right level of detail for any zoom and any viewport location.

## Setup — Load All LOD Files and Build All Indexes

We build one `GridIndex` per LOD level at startup. This is the one-time cost.

In [ ]:
import json
import math
import time
from pathlib import Path

lod_dir = Path("../../data/lod")

LOD_FILES = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

print("Loading LOD files...")
lod_features = {}
for name, filename in LOD_FILES.items():
    with open(lod_dir / filename) as f:
        lod_features[name] = json.load(f)["features"]
    print(f"  {name:<12} {len(lod_features[name]):>6,} features")

In [ ]:
def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]


class GridIndex:
    def __init__(self, cell_size=10.0):
        self.cell_size = cell_size
        self.cells = {}

    def _cells_for_bbox(self, bbox):
        lon_min, lat_min, lon_max, lat_max = bbox
        col_min = int((lon_min + 180) / self.cell_size)
        col_max = int((lon_max + 180) / self.cell_size)
        row_min = int((lat_min +  90) / self.cell_size)
        row_max = int((lat_max +  90) / self.cell_size)
        return [
            (col, row)
            for col in range(col_min, col_max + 1)
            for row in range(row_min, row_max + 1)
        ]

    def build(self, features):
        self.cells = {}
        for idx, feature in enumerate(features):
            for cell in self._cells_for_bbox(feature_bbox(feature)):
                if cell not in self.cells:
                    self.cells[cell] = []
                self.cells[cell].append((idx, feature))

    def query(self, viewport_bbox):
        seen = set()
        results = []
        for cell in self._cells_for_bbox(viewport_bbox):
            for idx, feature in self.cells.get(cell, []):
                if idx not in seen:
                    seen.add(idx)
                    results.append(feature)
        return results

In [ ]:
print("Building grid indexes...")
lod_indexes = {}
for name, features in lod_features.items():
    t0 = time.perf_counter()
    idx = GridIndex(cell_size=10.0)
    idx.build(features)
    elapsed = time.perf_counter() - t0
    lod_indexes[name] = idx
    print(f"  {name:<12} built in {elapsed:.3f}s")

print("\nAll indexes ready.")

## The Decision and Utility Functions

In [ ]:
def get_lod(zoom):
    z = int(math.floor(zoom))
    if z <= 3:  return "coarse"
    if z <= 6:  return "medium"
    if z <= 10: return "fine"
    return "extra_fine"


def leaflet_bounds_to_bbox(bounds):
    (lat_min, lon_min), (lat_max, lon_max) = bounds
    return [lon_min, lat_min, lon_max, lat_max]

## The Live Map

Two event handlers wire everything together:

- `on_zoom_change` — fires when zoom changes, switches to the correct LOD index, then re-queries
- `on_bounds_change` — fires when the user pans, re-queries the current LOD index

Both update the same layer, so there is only ever one GeoJSON layer on the map.

In [ ]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

# ── state ────────────────────────────────────────────────────────────────────
current_lod = get_lod(5)   # start at zoom 5

# ── map setup ────────────────────────────────────────────────────────────────
m = Map(center=[48.5, 10.0], zoom=5)

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)
m.add(layer)

# ── status widgets ────────────────────────────────────────────────────────────
lod_label     = widgets.Label(value="LOD: —")
feature_label = widgets.Label(value="Features: —")
time_label    = widgets.Label(value="Query: —")
status_bar    = widgets.HBox([lod_label, feature_label, time_label])

# ── update function ───────────────────────────────────────────────────────────
def update(*args):
    global current_lod

    if not m.bounds:
        return

    # Decide which LOD to use
    new_lod = get_lod(m.zoom)
    if new_lod != current_lod:
        current_lod = new_lod

    # Query the correct grid index
    vp = leaflet_bounds_to_bbox(m.bounds)
    t0 = time.perf_counter()
    visible = lod_indexes[current_lod].query(vp)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    # Update the layer
    layer.data = {"type": "FeatureCollection", "features": visible}

    # Update status
    lod_label.value     = f"LOD: {current_lod}"
    feature_label.value = f"  Features: {len(visible):,}"
    time_label.value    = f"  Query: {elapsed_ms:.2f}ms"

# ── wire events ───────────────────────────────────────────────────────────────
m.observe(update, names=["zoom", "bounds"])
update()   # initial render

widgets.VBox([m, status_bar])

**Try it:**
- Zoom out to 2 — the LOD status switches to `coarse` and feature count drops
- Zoom into a city — the LOD switches to `fine` or `extra_fine` and feature count drops (culling)
- Pan around at a fixed zoom — feature count updates as different regions come into view

The query time in the status bar shows how fast each lookup is.

## What We Just Built

Let's be explicit about the system:

```
User interaction
      │
      ▼
zoom / pan event
      │
      ├──► get_lod(zoom)  ──────► select correct GridIndex
      │
      └──► leaflet_bounds_to_bbox(bounds)
                │
                ▼
          index.query(viewport_bbox)
                │
                ▼
          visible features (deduplicated)
                │
                ▼
          update GeoJSON layer
```

Every component was built from scratch across these five modules:
- The simplified LOD files (Module 02)
- The bounding box intersection test (Module 03)
- The grid spatial index (Module 04)
- The zoom decision function (Module 05)

## Exercise A

Add a **zoom indicator** to the status bar that shows the current zoom level and a simple text label of the geographic scale (e.g. `zoom 5 — country scale`).

Use the zoom-to-scale table from Notebook 00 as a guide.

In [ ]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

def approx_viewport_width_deg(zoom):
    return 360 / (2 ** int(math.floor(zoom)))

current_lod = get_lod(5)
m2 = Map(center=[48.5, 10.0], zoom=5)

layer2 = GeoJSON(
    data={'type': 'FeatureCollection', 'features': []},
    style={'color': '#1d3557', 'weight': 1.5, 'opacity': 0.8}
)
m2.add(layer2)

lod_label = widgets.Label(value='LOD: —')
zoom_label = widgets.Label(value='Zoom: —')
scale_label = widgets.Label(value='Width: —')
feature_label = widgets.Label(value='Features: —')
status_bar2 = widgets.HBox([lod_label, zoom_label, scale_label, feature_label])

def update_with_scale(*args):
    global current_lod
    if not m2.bounds:
        return
    current_lod = get_lod(m2.zoom)
    vp = leaflet_bounds_to_bbox(m2.bounds)
    visible = lod_indexes[current_lod].query(vp)
    layer2.data = {'type': 'FeatureCollection', 'features': visible}
    lod_label.value = f'LOD: {current_lod}'
    zoom_label.value = f'  Zoom: {int(math.floor(m2.zoom))}'
    scale_label.value = f'  Width: {approx_viewport_width_deg(m2.zoom):.2f}°'
    feature_label.value = f'  Features: {len(visible):,}'

m2.observe(update_with_scale, names=['zoom', 'bounds'])
update_with_scale()

widgets.VBox([m2, status_bar2])


## Exercise B

The `update()` function is called for both zoom and bounds changes — a single event handler covers both.

This means if the user zooms AND pans at the same time (which ipyleaflet reports as two rapid events), `update()` runs twice. The second call sees the final state and overwrites the first — so the result is correct, but redundant work was done.

Add a `print` statement inside `update()` that shows which property triggered the call (`zoom` or `bounds`). Then scroll/zoom the map and observe the sequence. Do zoom and bounds always fire together?

In [ ]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

current_lod = get_lod(5)
m_debug = Map(center=[48.5, 10.0], zoom=5)

layer_debug = GeoJSON(
    data={'type': 'FeatureCollection', 'features': []},
    style={'color': '#e63946', 'weight': 1.5, 'opacity': 0.8}
)
m_debug.add(layer_debug)

debug_status = widgets.Label(value='Watch the notebook output for zoom/bounds events')

def update_debug(change=None):
    global current_lod
    if change is not None:
        print(change['name'])
    if not m_debug.bounds:
        return
    current_lod = get_lod(m_debug.zoom)
    vp = leaflet_bounds_to_bbox(m_debug.bounds)
    visible = lod_indexes[current_lod].query(vp)
    layer_debug.data = {'type': 'FeatureCollection', 'features': visible}
    debug_status.value = f'LOD: {current_lod} | features: {len(visible):,}'

m_debug.observe(update_debug, names=['zoom', 'bounds'])
update_debug()

widgets.VBox([m_debug, debug_status])


## Check Your Understanding

The map builds **four** grid indexes at startup — one per LOD level. This takes a few seconds and uses memory.

An alternative design would build the index lazily — only when the user first reaches that zoom range. Describe the tradeoffs between eager (build all at startup) and lazy (build on first use) index construction for this specific application.

Eager construction gives predictable performance once the map is open because every zoom range is ready immediately, but it spends startup time and memory even for LODs the user may never visit. Lazy construction reduces that up-front cost, but it turns the first visit to each new zoom range into a blocking build step that can feel like lag.


## Next

In [Module 06 — Putting It All Together](../06-Putting_It_Together/README.md), we assemble a clean, well-organized viewer notebook and reflect on what we built — before seeing the library version in Module 07.